---
title: Tables
---

## Results
### Table 1

In [1]:
#| label: tbl-r-musprop
#| tbl-cap: Estimated MTC properties as well as metrics derived from these properties.
#| echo: false
#| eval: true

import os, sys, pickle
import numpy as np
import pandas as pd
from pathlib import Path

# Set directories
cwd = Path.cwd()
baseDir = cwd.parent
dataDir = os.path.join(baseDir,'data')
funcDir = os.path.join(baseDir,'analysis','functions')
sys.path.append(str(funcDir))

import hillmodel, stats

#%% Initialize dataframe
# Define the variable names and types
# varnames,z = [['Hill constant', 'a'], ['Hill constant', 'b']].T

varnames = [['arel',    '$a^{rel}$',        '-',        'Hill constant'], 
            ['brel',    '$b^{rel}$',        '-',        'Hill constant'], 
            ['fmax',    '$F_{CE}^{max}$',   'N',        'maximal isometric CE force'],
            ['kpee',    '$k_{PEE}$',        'N/mm<sup>2</sup>', 'PEE stiffness scaling factor'], 
            ['ksee',    '$k_{SEE}$',        'N/mm<sup>2</sup>', 'SEE stiffness scaling factor'], 
            ['lce_opt', '$L_{CE}^{opt}$',   'mm',       'CE optimum length'],
            ['lpee0',   '$L_{PEE}^0$',      'mm',       'PEE slack length'],
            ['lsee0',   '$L_{SEE}^0$',      'mm',       'SEE slack length'],
            ['tact',    r'$\tau_{act}$',    'ms',       'Calcium dynamics activation time constant'],
            ['tdeact',  r'$\tau_{deact}$',  'ms',       'Calcium dynamics deactivation time constant'],
            ['lmtcOpt', '$L_{MTC}^{opt}$',  'mm',       'MTC length yielding maximal isometric SEE force'],
            ['pmax',    '$P_{CE}^{max}$',   'mW',       'maximal instantaneous CE power'],
            ['tHRise',  '$t_{HRT}$',        'ms',       r"`half-rise time'"],
            ['vmax',    '$v_{CE}^{max}$',   'mm/s',     'maximal CE shortening velocity'],
            ['vceopt',  '$v_{CE}^{opt}$',   'mm/s',     'CE velocity at $P_{CE}^{max}$']]

variables,symbols,units,descriptions, = list(zip(*varnames))  # This transposes the list of lists

types = ['MTC properties'] * 10 +['Derived metrics']*5
muscles = ['GMe1', 'GMe2', 'GMe3']

# Create the column names dynamically
columns = ['type','Description','Symbol','Unit'] + muscles

# Initialize the data dict with placeholder values
data = {col: [] for col in columns}

#%%  Fill the DataFrame
num_vars = len(variables)
num_gmes = len(muscles)

for iMus,mus in enumerate(muscles):
    parFile = os.path.join(dataDir,mus,mus+'_IM.pkl')
    muspar,dataQRout, dataSRout, dataACTout = pickle.load(open(parFile, 'rb'))
    
    # Compute arel & brel
    muspar['arel'] = muspar['a']/muspar['fmax']
    muspar['brel'] = muspar['b']/muspar['lce_opt']
    
    # Compute helper variables (or whatever we'll call it)
    muspar['lmtcOpt'] = (muspar['fmax']/muspar['ksee'])**0.5 + muspar['lsee0'] + muspar['lce_opt'] # [m] MTC length yielding maximal isometric SEE force
    muspar['vmax'] = muspar['b']/muspar['a']*muspar['fmax'] # [m/s] maximal CE shortening velocity
    muspar['vcenorm'] = muspar['a']*((1+muspar['fmax']/muspar['a'])**0.5-1)/muspar['fmax'] # [ ] normalised CE velocity at which instantaneous CE power is maximal 
    muspar['vceopt'] = muspar['vcenorm']*muspar['vmax']
    fcenorm = muspar['vcenorm'] # [ ] normalised CE force at which instantaneous CE power is maximal 
    muspar['pmax'] = muspar['vcenorm']*fcenorm*muspar['vmax']*muspar['fmax'] # [W] value of maximal instantaneous CE power
    _,_,gamma05 = hillmodel.act_state(np.nan,1,muspar) # [ ] [ ] value of gamma at which q=0.5
    muspar['tHRise'] = -muspar['tact']*np.log(1-gamma05) # [s] "half-rise time"
    
    for iVar,(var,description,symbol,unit,vartype) in enumerate(zip(variables,descriptions,symbols,units,types)):
        vartype = types[iVar]
        
        # Append data for each variable and its corresponding type
        if iMus == 0:  # Only append var and type once per variable
            data['type'].append(vartype)
            data['Description'].append(description)
            data['Symbol'].append(symbol)
            data['Unit'].append(unit)
        
        # Fill the GMe columns
        if var in ['lce_opt', 'lpee0', 'lsee0', 'tact', 'tdeact', 'lmtcOpt', 'pmax', 'tHRise', 'vmax', 'vceopt']:
            value = muspar[var]*1e3
        elif var in ['kpee', 'ksee']:
            value = muspar[var]/1e3
        else:
            value = muspar[var]
        data[mus].append(str(stats.str_round(value,3)))  # Adjust the logic as needed
        
# Create the DataFrame
df = pd.DataFrame(data)
df.map(str)
fileName = 'tbl_musprop'
df.to_csv(dataDir+'/'+fileName+'.csv', index=False, header=['type','description',
                    'symbol', 'unit', 'GMe1','GMe2','GMe3e'])

#%% TeX table
from great_tables import GT
from gt_tex import make_latex, delete_rows, insert_rows

df_tex = df.copy()
df_tex = df_tex.drop('type', axis=1)

gt_table = (GT(df_tex)
    #.tab_stub(rowname_col="description", groupname_col="type")
    .cols_align(align='center') 
    .cols_align(align='left', columns=['Description'])
    .cols_label(GMe1='1',GMe2='2',GMe3='3')
)

# Transform to LateX table
latex_str = make_latex(gt_table.as_latex())
latex_str = delete_rows(latex_str, row_numbers=[0])
add_rows = {
    0: r"  \bfseries Description & \bfseries Symbol & \bfseries Unit & \bfseries Rat 1 & \bfseries Rat 2 & \bfseries Rat 3 \\ \hline",
    1: r"  \multicolumn{6}{|l|}{\itshape MTC properties} \\ \hline",
    12: r"  \multicolumn{6}{|l|}{\itshape Derived metrics} \\ \hline"
}

latex_str = insert_rows(latex_str, add_rows)

# Write to a .tex file
with open("tbl-r-musprop.tex", "w", encoding="utf-8") as f:
    f.write(latex_str) 

#%% Great table
df_tex = df.copy()
gt_table = (GT(df_tex)
    .tab_stub(rowname_col="Description", groupname_col="type")
    .tab_spanner(label='Rat', columns=['GMe1', 'GMe2', 'GMe3'])
    .cols_align(align='center') 
    .cols_align(align='left', columns=['Description'])
)
gt_table


GT(_tbl_data=               type                                      Description  \
0    MTC properties                                    Hill constant   
1    MTC properties                                    Hill constant   
2    MTC properties                       maximal isometric CE force   
3    MTC properties                     PEE stiffness scaling factor   
4    MTC properties                     SEE stiffness scaling factor   
5    MTC properties                                CE optimum length   
6    MTC properties                                 PEE slack length   
7    MTC properties                                 SEE slack length   
8    MTC properties        Calcium dynamics activation time constant   
9    MTC properties      Calcium dynamics deactivation time constant   
10  Derived metrics  MTC length yielding maximal isometric SEE force   
11  Derived metrics                   maximal instantaneous CE power   
12  Derived metrics                                 `half-rise time'   
13  Derived metrics                   maximal CE shortening velocity   
14  Derived metrics                    CE velocity at $P_{CE}^{max}$   

             Symbol              Unit   GMe1   GMe2   GMe3  
0         $a^{rel}$                 -  0.569  0.712  0.649  
1         $b^{rel}$                 -   5.97   7.15   6.80  
2    $F_{CE}^{max}$                 N   15.5   14.1   17.0  
3         $k_{PEE}$  N/mm<sup>2</sup>   28.7   24.1   34.8  
4         $k_{SEE}$  N/mm<sup>2</sup>    952   1255   1050  
5    $L_{CE}^{opt}$                mm   13.7   14.7   13.3  
6       $L_{PEE}^0$                mm   15.1   15.6   14.5  
7       $L_{SEE}^0$                mm   28.8   29.3   25.2  
8      $\tau_{act}$                ms   55.2   57.7   41.6  
9    $\tau_{deact}$                ms   27.1   25.3   22.4  
10  $L_{MTC}^{opt}$                mm   46.6   47.3   42.5  
11   $P_{CE}^{max}$                mW    316    319    352  
12        $t_{HRT}$                ms   13.2   13.8   9.97  
13   $v_{CE}^{max}$              mm/s    144    147    140  
14   $v_{CE}^{opt}$              mm/s   54.1   57.8   53.8  , _body=<great_tables._gt_data.Body object at 0x7349dba97360>, _boxhead=Boxhead([ColInfo(var='type', type=<ColInfoTypeEnum.row_group: 3>, column_label='type', column_align='center', column_width=None), ColInfo(var='Description', type=<ColInfoTypeEnum.stub: 2>, column_label='Description', column_align='left', column_width=None), ColInfo(var='Symbol', type=<ColInfoTypeEnum.default: 1>, column_label='Symbol', column_align='center', column_width=None), ColInfo(var='Unit', type=<ColInfoTypeEnum.default: 1>, column_label='Unit', column_align='center', column_width=None), ColInfo(var='GMe1', type=<ColInfoTypeEnum.default: 1>, column_label='GMe1', column_align='center', column_width=None), ColInfo(var='GMe2', type=<ColInfoTypeEnum.default: 1>, column_label='GMe2', column_align='center', column_width=None), ColInfo(var='GMe3', type=<ColInfoTypeEnum.default: 1>, column_label='GMe3', column_align='center', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x7349dbaf2e90>, _spanners=Spanners([SpannerInfo(spanner_id='Rat', spanner_level=0, spanner_label='Rat', spanner_units=None, spanner_pattern=None, vars=['GMe1', 'GMe2', 'GMe3'], built=None)]), _heading=Heading(title=None, subtitle=None, preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x7349dbaf3390>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x7349dba97490>, _source_notes=[], _footnotes=[], _styles=[], _locale=<great_tables._gt_data.Locale object at 0x7349dbaf34d0>, _formats=[], _substitutions=[], _options=Options(table_id=OptionsInfo(scss=False, category='table', type='value', value=None), table_caption=OptionsInfo(scss=False, category='table', type='value', value=None), table_width=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_layout=OptionsInfo(scss=True, category='table', typ

## Supplementary material
### Table S1

In [2]:
#| label: supptbl-sscpa
#| tbl-cap: SSC parameters, stimulation durations, and measured AMPO of experimental stretch-shortening
#|   cycles with a 4 mm MTC length excursion. Stimulation onset was set at the start of MTC shortening in
#|   all conditions.

#%% Load packages & set directories
import os, sys
import numpy as np
import pandas as pd
from great_tables import GT, style, loc
from pathlib import Path

# Set directories
cwd = Path.cwd()
baseDir = cwd.parent
dataDir = os.path.join(baseDir,'data')
funcDir = os.path.join(baseDir,'analysis','functions')
sys.path.append(str(funcDir))

import stats , stimulation

#%% Set-up
exp = 'SSC_PA'
muscles = ['GMe1', 'GMe2', 'GMe3']

#%% Compute data values
# Motion parameters
if exp == 'SSC_PA':
    cf = np.array([1, 2, 3, 4, 5, 3, 3, 3, 3, 5, 4, 2, 1])
elif exp == 'SSC_PB':
    cf = np.array([1, 1.5, 2, 2.5, 3, 2, 2, 2, 2, 3, 2.5, 1.5, 1])

fts = np.array([0.50, 0.50, 0.50, 0.50, 0.50, 0.80, 0.65, 0.35, 0.20, 0.80, 0.65, 0.35, 0.20])
tShort = fts / cf * 1e3
tLeng = (1 - fts) / cf * 1e3

# Stimulation duration trial 1
iSuperscript = 1  
iTrial = 1
durStim1 = np.empty((len(muscles),len(cf)))
for iMus, mus in enumerate(muscles):      
    filepaths = [os.path.join(dataDir,mus,'dataExp',exp,f'{mus}_{exp}{iCond:02d}_{iTrial:01d}.csv') for iCond in range(1,14)]
    durStim  = stimulation.get_stim_dur(filepaths)
    durStim1[iMus,:] = [x*1e3 for x in durStim] # to ms

durStim1_str = [] 
for iCond in range(1,len(cf)+1):
    same,diff,i = stats.analyse_3similar(durStim1[:,iCond-1],1)
    if i == True:
        durStim1_str.append(stats.str_round(same,2))
    else:
        durStim1_str.append(stats.str_round(same,2)+f'<sup>{iSuperscript}</sup>')
        #print(f'{iSuperscript}: Cond {iCond:02d}, GMe{i+1} stimDuration = {diff:0.0f} ms')
        iSuperscript +=1
        
# Stimulation duration trial 2
iSuperscript = 1  
iTrial = 2
durStim2 = np.empty((len(muscles),len(cf)))
for iMus, mus in enumerate(muscles):   
    filepaths = [os.path.join(dataDir,mus,'dataExp',exp,f'{mus}_{exp}{iCond:02d}_{iTrial:01d}.csv') for iCond in range(1,14)]
    durStim  = stimulation.get_stim_dur(filepaths)
    durStim2[iMus,:] = [x*1e3 for x in durStim] # to ms

durStim2_str = [] 
for iCond in range(1,len(cf)+1):
    same,diff,i = stats.analyse_3similar(durStim2[:,iCond-1],1)
    if i == True:
        durStim2_str.append(stats.str_round(same,2))
    else:
        durStim2_str.append(stats.str_round(same,2)+f'<sup>{iSuperscript}</sup>')
        #print(f'{iSuperscript}: Cond {iCond:02d}, GMe{i+1} stimDuration = {diff:0.0f} ms')
        iSuperscript +=1
        
# AMPO of the rats:
AMPO = []
for mus in ['GMe1', 'GMe2', 'GMe3']:
    fileName = mus+'_dataAMPO'
    df = pd.read_excel(dataDir+'/'+mus+'/'+fileName+'.xlsx')
    ampoData = df.to_numpy()
    
    if exp == 'SSC_PA':
        t1 = np.mean(ampoData[0:3,5:],0)
        t2 = np.mean(ampoData[3:6,5:],0)
    elif exp == 'SSC_PB':
        t1 = np.mean(ampoData[6:9,5:],0)
        t2 = np.mean(ampoData[9:12,5:],0)
    t1 = [stats.str_round(x,2) for x in t1]
    t2 = [stats.str_round(x,2) for x in t2]
    
    AMPO.append(t1)
    AMPO.append(t2)
AMPO = np.array(AMPO)

#%%
# First 4 rows are calculated values; rest are placeholders (NaN)
data_values = np.full((12, len(cf)), np.nan, dtype=object)
data_values[0] = [stats.str_round(x,2) for x in cf]
data_values[1] = [stats.str_round(x,2) for x in fts]
data_values[2] = [stats.str_round(x,3) for x in tShort]
data_values[3] = [stats.str_round(x,3) for x in tLeng]
data_values[4] = durStim1_str
data_values[5] = durStim2_str
data_values[6:] = AMPO

# === Create DataFrame ===
descriptions = [
    'Cycle frequency', 'FTS', 'MTC shortening time', 'MTC lengthening time',
    'Trial 1', 'Trial 2',
    'Trial 1', 'Trial 2',
    'Trial 1', 'Trial 2',
    'Trial 1', 'Trial 2',
]

units = ['Hz', '-', 'ms', 'ms', 'ms', 'ms', 'mW', 'mW', 'mW', 'mW', 'mW', 'mW']
types = ['SSC parameters'] * 2 + ['MTC shortening and lengthening times'] * 2 + ['Stimulation durations'] * 2 + ['Measured AMPO of rat 1'] * 2 + ['Measured AMPO of rat 2'] * 2 + ['Measured AMPO of rat 3'] * 2
conds = [str(i) for i in range(1, 14)]

df = pd.DataFrame(
    data=np.column_stack([types, descriptions, units, data_values]),
    columns=['type', 'Description', 'Unit'] + conds
)

#%% TeX table
from great_tables import GT
from gt_tex import make_latex, insert_rows, fix_reference, replace_latex_table_cell, delete_rows, replace_superscripts

df_tex = df.copy()
df_tex = df_tex.drop('type', axis=1)

gt_table = (GT(df_tex)
    #.tab_stub(rowname_col="description", groupname_col="type")
    .cols_align(align='center') 
    .cols_align(align='left', columns=['Description'])
    .cols_label(Description='')
)

latex_str = make_latex(gt_table.as_latex())
add_rows = {
    0: r"  & & \multicolumn{13}{c|}{Condition}  \\ \hline",
    1: r"  \bfseries & \bfseries Unit & \bfseries 1 & \bfseries 2 & \bfseries 3 & \bfseries 4 & \bfseries 5 & \bfseries 6 & \bfseries 7 & \bfseries 8 & \bfseries 9 & \bfseries 10 & \bfseries 11 & \bfseries 12 & \bfseries 13 \\ \hline",
    2: r"  \multicolumn{15}{|l|}{\itshape SSC parameters} \\ \hline",
    5: r"  \multicolumn{15}{|l|}{\itshape MTC shortening and lengthening times} \\ \hline",
    8: r"  \multicolumn{15}{|l|}{\itshape Stimulation durations} \\ \hline",
    11: r"  \multicolumn{15}{|l|}{\itshape Measured AMPO of Rat 1} \\ \hline",
    14: r"  \multicolumn{15}{|l|}{\itshape Measured AMPO of Rat 2} \\ \hline",
    17: r"  \multicolumn{15}{|l|}{\itshape Measured AMPO of Rat 3} \\ \hline",
}
latex_str = delete_rows(latex_str, row_numbers=[0])
latex_str = insert_rows(latex_str, add_rows)

latex_str = replace_latex_table_cell(latex_str, row=8, col=0, new_text=r'Trial 1')
latex_str = replace_latex_table_cell(latex_str, row=9, col=0, new_text=r'Trial 2')
latex_str = replace_latex_table_cell(latex_str, row=10, col=0, new_text=r'Trial 1')
latex_str = replace_latex_table_cell(latex_str, row=11, col=0, new_text=r'Trial 2')
latex_str = replace_latex_table_cell(latex_str, row=12, col=0, new_text=r'Trial 1')
latex_str = replace_latex_table_cell(latex_str, row=13, col=0, new_text=r'Trial 2')

# Write to a .tex file
latex_str += (r"\break\hfill\footnotesize{"+ 
              r"\textsuperscript{1} For conditon 1: stimulation duration of rat 3 was 455 ms. "
              r"\textsuperscript{2} For conditon 9: stimulation duration of rat 3 was 23 ms. "
              r"\textsuperscript{3} For conditon 13: stimulation duration of rat 1 was 165 ms.}")

with open('supptbl-sscpa.tex', "w", encoding="utf-8") as f:
    f.write(latex_str)


#%% Great table
from great_tables import GT, md
df_gt = df.copy()

gt_table = (GT(df_gt)
    .tab_spanner(label = "Condition", columns = [f'{x}' for x in range(1,14)])
    .tab_stub(rowname_col="Description", groupname_col="type")
    .tab_style(style = style.text(style = "italic"), locations = loc.row_groups())
    .tab_source_note(
        source_note = md("<sup>1</sup> For conditon 1: stimulation duration of rat 3 was 455 ms.")
    )
    .tab_source_note(
        source_note = md("<sup>2</sup> For conditon 9: stimulation duration of rat 3 was 23 ms.")
    )
    .tab_source_note(
        source_note = md("<sup>3</sup> For conditon 13: stimulation duration of rat 1 was 165 ms.")
    )
)
gt_table




GT(_tbl_data=                                    type           Description Unit  \
0                         SSC parameters       Cycle frequency   Hz   
1                         SSC parameters                   FTS    -   
2   MTC shortening and lengthening times   MTC shortening time   ms   
3   MTC shortening and lengthening times  MTC lengthening time   ms   
4                  Stimulation durations               Trial 1   ms   
5                  Stimulation durations               Trial 2   ms   
6                 Measured AMPO of rat 1               Trial 1   mW   
7                 Measured AMPO of rat 1               Trial 2   mW   
8                 Measured AMPO of rat 2               Trial 1   mW   
9                 Measured AMPO of rat 2               Trial 2   mW   
10                Measured AMPO of rat 3               Trial 1   mW   
11                Measured AMPO of rat 3               Trial 2   mW   

                  1     2     3     4     5     6     7     8               9  \
0               1.0   2.0   3.0   4.0   5.0   3.0   3.0   3.0             3.0   
1              0.50  0.50  0.50  0.50  0.50  0.80  0.65  0.35            0.20   
2               500   250   167   125   100   267   217   117            66.7   
3               500   250   167   125   100  66.7   117   217             267   
4               405   175   103    65    35   183   133    53              23   
5   445<sup>1</sup>   205   123    85    55   213   163    73  33<sup>2</sup>   
6                31    51    59    58    45    74    67    38              20   
7                34    56    69    72    66    84    77    49              25   
8                29    48    57    58    45    71    62    37              18   
9                30    52    65    68    64    78    72    46              24   
10               34    56    65    66    48    83    72    44              22   
11               37    62    74    82    74    91    85    56              20   

      10    11    12               13  
0    5.0   4.0   2.0              1.0  
1   0.80  0.65  0.35             0.20  
2    160   162   175              200  
3   40.0  87.5   325              800  
4     85    95   115              135  
5    115   115   135  155<sup>3</sup>  
6     82    74    43               23  
7    104    87    47               27  
8     79    70    40               22  
9     93    79    45               24  
10    91    82    47               25  
11   116    96    54               28  , _body=<great_tables._gt_data.Body object at 0x7349d9124e50>, _boxhead=Boxhead([ColInfo(var='type', type=<ColInfoTypeEnum.row_group: 3>, column_label='type', column_align='left', column_width=None), ColInfo(var='Description', type=<ColInfoTypeEnum.stub: 2>, column_label='Description', column_align='left', column_width=None), ColInfo(var='Unit', type=<ColInfoTypeEnum.default: 1>, column_label='Unit', column_align='left', column_width=None), ColInfo(var='1', type=<ColInfoTypeEnum.default: 1>, column_label='1', column_align='left', column_width=None), ColInfo(var='2', type=<ColInfoTypeEnum.default: 1>, column_label='2', column_align='right', column_width=None), ColInfo(var='3', type=<ColInfoTypeEnum.default: 1>, column_label='3', column_align='right', column_width=None), ColInfo(var='4', type=<ColInfoTypeEnum.default: 1>, column_label='4', column_align='right', column_width=None), ColInfo(var='5', type=<ColInfoTypeEnum.default: 1>, column_label='5', column_align='right', column_width=None), ColInfo(var='6', type=<ColInfoTypeEnum.default: 1>, column_label='6', column_align='right', column_width=None), ColInfo(var='7', type=<ColInfoTypeEnum.default: 1>, column_label='7', column_align='right', column_width=None), ColInfo(var='8', type=<ColInfoTypeEnum.default: 1>, column_label='8', column_align='right', column_width=None), ColInfo(var='9', type=<ColInfoTypeEnum.default: 1>, column_label='9', column_align='left', column_width=None), ColInfo(var='10', type=<ColInfoTypeEnum

### Table S2

In [3]:
#| label: supptbl-sscpb
#| tbl-cap: SSC parameters, stimulation durations, and measured AMPO of experimental stretch-shortening
#|   cycles with an 8 mm MTC length excursion. Stimulation onset was set at the start of MTC shortening in
#|   all conditions.

#%% Load packages & set directories
import os, sys
import numpy as np
import pandas as pd
from great_tables import GT, style, loc
from pathlib import Path

# Set directories
cwd = Path.cwd()
baseDir = cwd.parent
dataDir = os.path.join(baseDir,'data')
funcDir = os.path.join(baseDir,'analysis','functions')
sys.path.append(str(funcDir))

import stats, stimulation

#%% Set-up
exp = 'SSC_PB'
muscles = ['GMe1', 'GMe2', 'GMe3']

#%% Compute data values
# Motion parameters
if exp == 'SSC_PA':
    cf = np.array([1, 2, 3, 4, 5, 3, 3, 3, 3, 5, 4, 2, 1])
elif exp == 'SSC_PB':
    cf = np.array([1, 1.5, 2, 2.5, 3, 2, 2, 2, 2, 3, 2.5, 1.5, 1])

fts = np.array([0.50, 0.50, 0.50, 0.50, 0.50, 0.80, 0.65, 0.35, 0.20, 0.80, 0.65, 0.35, 0.20])
tShort = fts / cf * 1e3
tLeng = (1 - fts) / cf * 1e3

# Stimulation duration trial 1
iSuperscript = 1  
iTrial = 1
durStim1 = np.empty((len(muscles),len(cf)))
for iMus, mus in enumerate(muscles):      
    filepaths = [os.path.join(dataDir,mus,'dataExp',exp,f'{mus}_{exp}{iCond:02d}_{iTrial:01d}.csv') for iCond in range(1,14)]
    durStim  = stimulation.get_stim_dur(filepaths)
    durStim1[iMus,:] = [x*1e3 for x in durStim] # to ms

durStim1_str = [] 
for iCond in range(1,len(cf)+1):
    same,diff,i = stats.analyse_3similar(durStim1[:,iCond-1],1)
    if i == True:
        durStim1_str.append(stats.str_round(same,2))
    else:
        durStim1_str.append(stats.str_round(same,2)+f'<sup>{iSuperscript}</sup>')
        #print(f'{iSuperscript}: Cond {iCond:02d}, GMe{i+1} stimDuration = {diff:0.0f} ms')
        iSuperscript +=1
        
# Stimulation duration trial 2
iSuperscript = 1  
iTrial = 2
durStim2 = np.empty((len(muscles),len(cf)))
for iMus, mus in enumerate(muscles):   
    filepaths = [os.path.join(dataDir,mus,'dataExp',exp,f'{mus}_{exp}{iCond:02d}_{iTrial:01d}.csv') for iCond in range(1,14)]
    durStim  = stimulation.get_stim_dur(filepaths)
    durStim2[iMus,:] = [x*1e3 for x in durStim] # to ms

durStim2_str = [] 
for iCond in range(1,len(cf)+1):
    same,diff,i = stats.analyse_3similar(durStim2[:,iCond-1],1)
    if i == True:
        durStim2_str.append(stats.str_round(same,2))
    else:
        durStim2_str.append(stats.str_round(same,2)+f'<sup>{iSuperscript}</sup>')
        #print(f'{iSuperscript}: Cond {iCond:02d}, GMe{i+1} stimDuration = {diff:0.0f} ms')
        iSuperscript +=1
        
# AMPO of the rats:
AMPO = []
for mus in ['GMe1', 'GMe2', 'GMe3']:
    fileName = mus+'_dataAMPO'
    df = pd.read_excel(dataDir+'/'+mus+'/'+fileName+'.xlsx')
    ampoData = df.to_numpy()
    
    if exp == 'SSC_PA':
        t1 = np.mean(ampoData[0:3,5:],0)
        t2 = np.mean(ampoData[3:6,5:],0)
    elif exp == 'SSC_PB':
        t1 = np.mean(ampoData[6:9,5:],0)
        t2 = np.mean(ampoData[9:12,5:],0)
    t1 = [stats.str_round(x,2) for x in t1]
    t2 = [stats.str_round(x,2) for x in t2]
    
    AMPO.append(t1)
    AMPO.append(t2)
AMPO = np.array(AMPO)

#%%
# First 4 rows are calculated values; rest are placeholders (NaN)
data_values = np.full((12, len(cf)), np.nan, dtype=object)
data_values[0] = [stats.str_round(x,2) for x in cf]
data_values[1] = [stats.str_round(x,2) for x in fts]
data_values[2] = [stats.str_round(x,3) for x in tShort]
data_values[3] = [stats.str_round(x,3) for x in tLeng]
data_values[4] = durStim1_str
data_values[5] = durStim2_str
data_values[6:] = AMPO

# === Create DataFrame ===
descriptions = [
    'Cycle frequency', 'FTS', 'MTC shortening time', 'MTC lengthening time',
    'Trial 1', 'Trial 2',
    'Trial 1', 'Trial 2',
    'Trial 1', 'Trial 2',
    'Trial 1', 'Trial 2',
]

units = ['Hz', '-', 'ms', 'ms', 'ms', 'ms', 'mW', 'mW', 'mW', 'mW', 'mW', 'mW']
types = ['SSC parameters'] * 2 + ['MTC shortening and lengthening times'] * 2 + ['Stimulation durations'] * 2 + ['Measured AMPO of rat 1'] * 2 + ['Measured AMPO of rat 2'] * 2 + ['Measured AMPO of rat 3'] * 2
conds = [str(i) for i in range(1, 14)]

df = pd.DataFrame(
    data=np.column_stack([types, descriptions, units, data_values]),
    columns=['type', 'Description', 'Unit'] + conds
)

#%% TeX table
from great_tables import GT
from gt_tex import make_latex, insert_rows, fix_reference, replace_latex_table_cell, delete_rows, replace_superscripts

df_tex = df.copy()
df_tex = df_tex.drop('type', axis=1)

gt_table = (GT(df_tex)
    #.tab_stub(rowname_col="description", groupname_col="type")
    .cols_align(align='center') 
    .cols_align(align='left', columns=['Description'])
    .cols_label(Description='')
)

latex_str = make_latex(gt_table.as_latex())
add_rows = {
    0: r"  & & \multicolumn{13}{c|}{Condition}  \\ \hline",
    1: r"  \bfseries & \bfseries Unit & \bfseries 1 & \bfseries 2 & \bfseries 3 & \bfseries 4 & \bfseries 5 & \bfseries 6 & \bfseries 7 & \bfseries 8 & \bfseries 9 & \bfseries 10 & \bfseries 11 & \bfseries 12 & \bfseries 13 \\ \hline",
    2: r"  \multicolumn{15}{|l|}{\itshape SSC parameters} \\ \hline",
    5: r"  \multicolumn{15}{|l|}{\itshape MTC shortening and lengthening times} \\ \hline",
    8: r"  \multicolumn{15}{|l|}{\itshape Stimulation durations} \\ \hline",
    11: r"  \multicolumn{15}{|l|}{\itshape Measured AMPO of Rat 1} \\ \hline",
    14: r"  \multicolumn{15}{|l|}{\itshape Measured AMPO of Rat 2} \\ \hline",
    17: r"  \multicolumn{15}{|l|}{\itshape Measured AMPO of Rat 3} \\ \hline",
}
latex_str = delete_rows(latex_str, row_numbers=[0])
latex_str = insert_rows(latex_str, add_rows)

latex_str = replace_latex_table_cell(latex_str, row=8, col=0, new_text=r'Trial 1')
latex_str = replace_latex_table_cell(latex_str, row=9, col=0, new_text=r'Trial 2')
latex_str = replace_latex_table_cell(latex_str, row=10, col=0, new_text=r'Trial 1')
latex_str = replace_latex_table_cell(latex_str, row=11, col=0, new_text=r'Trial 2')
latex_str = replace_latex_table_cell(latex_str, row=12, col=0, new_text=r'Trial 1')
latex_str = replace_latex_table_cell(latex_str, row=13, col=0, new_text=r'Trial 2')

# Write to a .tex file
latex_str += (r"\break\hfill\footnotesize{"+ 
              r"\textsuperscript{1} For conditon 1: stimulation duration of rat 3 was 455 ms. "
              r"\textsuperscript{2} For conditon 9: stimulation duration of rat 3 was 23 ms. "
              r"\textsuperscript{3} For conditon 13: stimulation duration of rat 1 was 165 ms.}")

with open('supptbl-sscpb.tex', "w", encoding="utf-8") as f:
    f.write(latex_str)


#%% Great table
from great_tables import GT, md
df_gt = df.copy()

gt_table = (GT(df_gt)
    .tab_spanner(label = "Condition", columns = [f'{x}' for x in range(1,14)])
    .tab_stub(rowname_col="Description", groupname_col="type")
    .tab_style(style = style.text(style = "italic"), locations = loc.row_groups())
    .tab_source_note(
        source_note = md("<sup>1</sup> For conditon 1: stimulation duration of rat 3 was 455 ms.")
    )
    .tab_source_note(
        source_note = md("<sup>2</sup> For conditon 9: stimulation duration of rat 3 was 23 ms.")
    )
    .tab_source_note(
        source_note = md("<sup>3</sup> For conditon 13: stimulation duration of rat 1 was 165 ms.")
    )
)
gt_table




GT(_tbl_data=                                    type           Description Unit     1  \
0                         SSC parameters       Cycle frequency   Hz   1.0   
1                         SSC parameters                   FTS    -  0.50   
2   MTC shortening and lengthening times   MTC shortening time   ms   500   
3   MTC shortening and lengthening times  MTC lengthening time   ms   500   
4                  Stimulation durations               Trial 1   ms   405   
5                  Stimulation durations               Trial 2   ms   455   
6                 Measured AMPO of rat 1               Trial 1   mW    53   
7                 Measured AMPO of rat 1               Trial 2   mW    54   
8                 Measured AMPO of rat 2               Trial 1   mW    49   
9                 Measured AMPO of rat 2               Trial 2   mW    50   
10                Measured AMPO of rat 3               Trial 1   mW    57   
11                Measured AMPO of rat 3               Trial 2   mW    47   

       2     3     4     5      6     7     8               9  \
0    1.5   2.0   2.5   3.0    2.0   2.0   2.0             2.0   
1   0.50  0.50  0.50  0.50   0.80  0.65  0.35            0.20   
2    333   250   200   167    400   325   175             100   
3    333   250   200   167  100.0   175   325             400   
4    253   175   135   103    295   245   115              55   
5    293   205   165   123    345   275   135  75<sup>1</sup>   
6     67    75    80    79     94    89    55              26   
7     47    80    86    85     99    97    63              28   
8     63    73    75    76     88    83    52              24   
9     66    76    80    81     92    86    55              27   
10    73    81    87    88    103    95    60              28   
11     -    89     -    88    110     -     -              32   

                 10    11    12               13  
0               3.0   2.5   1.5              1.0  
1              0.80  0.65  0.35             0.20  
2               267   260   233              200  
3              66.7   140   433              800  
4               173   175   163              145  
5   213<sup>2</sup>   225   193  165<sup>3</sup>  
6               113    95    54               34  
7               121   104    62               35  
8               104    88    50               32  
9               113    95    54               33  
10              123   101    60               37  
11              129     -     -               40  , _body=<great_tables._gt_data.Body object at 0x7349d8fca5d0>, _boxhead=Boxhead([ColInfo(var='type', type=<ColInfoTypeEnum.row_group: 3>, column_label='type', column_align='left', column_width=None), ColInfo(var='Description', type=<ColInfoTypeEnum.stub: 2>, column_label='Description', column_align='left', column_width=None), ColInfo(var='Unit', type=<ColInfoTypeEnum.default: 1>, column_label='Unit', column_align='left', column_width=None), ColInfo(var='1', type=<ColInfoTypeEnum.default: 1>, column_label='1', column_align='right', column_width=None), ColInfo(var='2', type=<ColInfoTypeEnum.default: 1>, column_label='2', column_align='right', column_width=None), ColInfo(var='3', type=<ColInfoTypeEnum.default: 1>, column_label='3', column_align='right', column_width=None), ColInfo(var='4', type=<ColInfoTypeEnum.default: 1>, column_label='4', column_align='right', column_width=None), ColInfo(var='5', type=<ColInfoTypeEnum.default: 1>, column_label='5', column_align='right', column_width=None), ColInfo(var='6', type=<ColInfoTypeEnum.default: 1>, column_label='6', column_align='right', column_width=None), ColInfo(var='7', type=<ColInfoTypeEnum.default: 1>, column_label='7', column_align='right', column_width=None), ColInfo(var='8', type=<ColInfoTypeEnum.default: 1>, column_label='8', column_align='right', column_width=None), ColInfo(var='9', type=<ColInfoTypeEnum.default: 1>, column_label='9', column_align='left', column_width=None), ColInfo(var='10', type=<C